### Data Aggregation

Aggregate and prepare data for modeling.

In [2]:
import os
import pandas as pd

In [3]:
# stats files
stats = []
for file in os.listdir('../data/stats/consolidated/'):
    season = int(file[:4])
    temp = pd.read_excel('../data/stats/consolidated/' + file)
    temp['SEASON'] = season
    stats.append(temp)

# concatenate into one dataframe
stats = pd.concat(stats)
print(f'{stats.shape = }')

# salaries files
salaries = []
for file in os.listdir('../data/salaries/'):
    season = int(file[:4])
    temp = pd.read_excel('../data/salaries/' + file)
    temp['SEASON'] = season
    salaries.append(temp)

# concatenate into one dataframe
salaries = pd.concat(salaries)
print(f'{salaries.shape = }')

stats.shape = (72771, 39)
salaries.shape = (118258, 11)


In [4]:
# merge into one dataframe with season/week/player as key
dat = salaries.merge(
    stats, how='inner', # TODO review inner: do I want to filter out actual without a projection?
    left_on=['SEASON', 'WK', 'NAME'],
    right_on=['SEASON', 'WK', 'NAME'],
    indicator=True
)

print(f'{dat.shape = }')

# filter out duplicates (these show up in FantasyData's tables)
dat = dat.drop_duplicates(['SEASON', 'WK', 'NAME'])
print(f'{dat.shape = }')

# filter out bad salaries
dat = dat[dat['$']>1000]
print(f'{dat.shape = }')

dat.shape = (65656, 48)
dat.shape = (58787, 48)
dat.shape = (58770, 48)


In [5]:
# group data to rank by position projections and actuals
method = 'min'
grouped = dat.groupby(['SEASON', 'WK', 'POS_x'])
proj_rank = grouped['FPTS_x'].rank(ascending=False, method=method).astype(int)
act_rank = grouped['FPTS_y'].rank(ascending=False, method=method).astype(int)

# append to the dataframe
dat['rank_proj'] = proj_rank
dat['rank_actual'] = act_rank

# append tiers to dataframe
dat['tier_proj'] = proj_rank.apply(lambda x: 1 if x <= 4 else 2 if x <= 8 else 3 if x <= 12 else 4)
dat['tier_actual'] = act_rank.apply(lambda x: 1 if x <= 4 else 2 if x <= 8 else 3 if x <= 12 else 4)

In [13]:
feats = dat[[
    'NAME', 'SEASON', 'WK', # identifiers
    'TEAM_x', 'OPP_x', 'OPP RANK', # teams
    'POS_x', 'OPP POS RANK', # positions
    '$', 'FPTS_x', 'rank_proj', 'tier_proj', # projections
    'FPTS_y', 'rank_actual', 'tier_actual' # actuals
    ]]

feats.columns = [
    'name', 'season', 'week',
    'team', 'opp', 'opp_rank',
    'pos', 'opp_pos_rank',
    'salary', 'fpts_proj', 'rank_proj', 'tier_proj',
    'fpts_actual', 'rank_actual', 'tier_actual'
]

In [17]:
# feats.to_csv('feature_dataset.csv', index=False)